In [24]:
import cv2

video_path = "../data/walking.mp4"

cap = cv2.VideoCapture(video_path)

print("Openai:", cap.isOpened())

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = frame_count / fps

print("FPS:", fps)
print("Frames:", frame_count)
print("duration:",duration)

Openai: True
FPS: 30.0
Frames: 845
duration: 28.166666666666668


In [25]:
cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

print("Lecture reussie:", ret)
print("Type :",type((frame)))
print("shape :", frame.shape)

Lecture reussie: True
Type : <class 'numpy.ndarray'>
shape : (1080, 1920, 3)


In [26]:
import torch

In [27]:
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
frame_resized = cv2.resize(frame_rgb, (224, 224))

In [28]:
tensor_frame = torch.from_numpy(frame_resized)
tensor_frame = tensor_frame.permute(2, 0, 1)
print("Avant :", frame.shape)
print("Après :", tensor_frame.shape)

Avant : (1080, 1920, 3)
Après : torch.Size([3, 224, 224])


In [29]:
import numpy as np

In [30]:
indices = np.linspace(0, frame_count - 1, 16, dtype=int)
print(indices)

[  0  56 112 168 225 281 337 393 450 506 562 618 675 731 787 844]


In [31]:
cap = cv2.VideoCapture(video_path)

frames = []

for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))

    ret, frame = cap.read()

    if not ret:
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (224, 224))

    frame_tensor = torch.from_numpy(frame)
    frame_tensor = frame_tensor.permute(2, 0, 1)

    frames.append(frame_tensor)

cap.release()

len(frames)

16

In [32]:
video_tensor = torch.stack(frames)
print(video_tensor.shape)
print(len(frames))
print(video_tensor.dtype)
print(video_tensor.min(), video_tensor.max())

torch.Size([16, 3, 224, 224])
16
torch.uint8
tensor(0, dtype=torch.uint8) tensor(255, dtype=torch.uint8)


In [33]:
video_tensor =  video_tensor.float() / 255.0
print(video_tensor.dtype)
print(video_tensor.min(), video_tensor.max())

torch.float32
tensor(0.) tensor(1.)


## Creation d'une fonction

In [34]:
def load_video(video_path, num_frames=16, size=(224, 224)):
    cap = cv2.VideoCapture(video_path)

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    indices = np.linspace(
        0,
        frame_count - 1,
        num_frames,
        dtype=int
    )

    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))

        ret, frame = cap.read()

        if not ret:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, size)

        frame = torch.from_numpy(frame)
        frame = frame.permute(2, 0, 1)

        frames.append(frame)

    cap.release()

    video_tensor = torch.stack(frames).float() / 255.0

    return video_tensor

In [35]:
video = load_video("../data/walking.mp4")

print(video.shape)
print(video.dtype)
print(video.min(), video.max())

torch.Size([16, 3, 224, 224])
torch.float32
tensor(0.) tensor(1.)
